In [2]:
# ----------------- Inference & Evaluation -----------------
def run_inference(model, tokenizer, dataset, max_len=128, num_samples=100):
    model.eval()
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer.src_lang = "kk"
    
    bleu = evaluate.load("sacrebleu")
    chrf = evaluate.load("chrf")

    preds = []
    refs = []
    source=[]

    for example in tqdm(dataset.select(range(num_samples)), desc="🔹 Running inference"):
        inputs = tokenizer(example["kazakh"], return_tensors="pt", truncation=True, padding=True, max_length=max_len).to(model.device)
        with torch.no_grad():
            generated = model.generate(**inputs, forced_bos_token_id=tokenizer.get_lang_id("ru"), max_length=max_len)
        pred = tokenizer.decode(generated[0], skip_special_tokens=True)
        ref = example["russian"]

        preds.append(pred)
        refs.append([ref])  # BLEU expects list of references
        source.append(example["kazakh"])

        bleu.add(prediction=pred, reference=[ref])
        chrf.add(prediction=pred, reference=ref)

    bleu_score = bleu.compute()
    chrf_score = chrf.compute()

    print("🔹 BLEU:", bleu_score)
    print("🔹 chrF++:", chrf_score)

    return preds, refs, source

In [3]:
from train import load_data

import os
import pandas as pd
import polars as pl
import torch
import evaluate
from datasets import Dataset
import numpy as np
from transformers import (
    M2M100Tokenizer,
    M2M100ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from tqdm import tqdm

model_dir='/home/lilo/experiments/exp_m2m100_065_kz_rus'
test='/home/lilo/cleaned_data/test_dedup.txt'

test_set = load_data(test, 0.65)
tokenizer = M2M100Tokenizer.from_pretrained(model_dir)
model = M2M100ForConditionalGeneration.from_pretrained(model_dir)


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔹 Using GPU: 0 (NVIDIA GeForce RTX 3090)


In [3]:
model.device

device(type='cpu')

In [32]:
33.83-30.74

3.09

In [12]:
from train import tokenize_data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)
dataset = tokenize_data(test_set, tokenizer, max_length=112, cache_dir='./cache')

preds, ref, source = run_inference(model, tokenizer, dataset, max_len=112, num_samples=1000) ###set len(test_set) for full metrics

🔹 Tokenizing dataset with cache file: ./cache/tokenized_cache.arrow


🔹 Running inference: 100%|██████████| 1000/1000 [03:54<00:00,  4.27it/s]


🔹 BLEU: {'score': 30.743320011339637, 'counts': [7995, 4859, 3390, 2457], 'totals': [13814, 12814, 11852, 10974], 'precisions': [57.87606775734762, 37.919463087248324, 28.602767465406682, 22.38928376161837], 'bp': 0.8928909122158043, 'sys_len': 13814, 'ref_len': 15379}
🔹 chrF++: {'score': 51.863807137906804, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [5]:
len(preds)

1000

In [8]:
# i=679
print(ref[i][0], '\n', preds[i], '\n', source[i])
i+=1

46) приобретения электроэнергии; 
 46) приобретение электроэнергии; 
 46) электр энергиясын сатып алу;


In [17]:
# i=679
print(ref[i][0], '\n', preds[i], '\n', source[i])
i+=1

В научно-исследовательской работе Института принимают активное участие руководство института, сотрудники факультета очного обучения, факультета повышения квалификации, кафедры института. 
 В научно-исследовательскую работу института активно участвуют руководство института, сотрудники факультета повседневного обучения, факультета профессионального и дополнительного образования, кафедр института. 
 Институттың ғылыми-зерттеу жұмысына институт басшылығы, күндізгі оқыту факультетінің, кәсіби және қосымша білім беру факультетінің, институт кафедраларының қызметкерлері белсене араласады.


filtered res

In [18]:
from train import load_data

import os
import pandas as pd
import polars as pl
import torch
import evaluate
from datasets import Dataset
import numpy as np
from transformers import (
    M2M100Tokenizer,
    M2M100ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from tqdm import tqdm

model_dir='/home/lilo/experiments/exp_m2m100_filtered065_kz_rus'
test='/home/lilo/cleaned_data/test_dedup.txt'

test_set = load_data(test, 0.65)
tokenizer = M2M100Tokenizer.from_pretrained(model_dir)
model = M2M100ForConditionalGeneration.from_pretrained(model_dir)


In [ ]:
from train import tokenize_data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)
dataset = tokenize_data(test_set, tokenizer, max_length=112, cache_dir=None)

predsf, reff, sourcef = run_inference(model, tokenizer, dataset, max_len=112, num_samples=1000) ###set len(test_set) for full metrics

Map:  80%|████████  | 50688/63081 [00:15<00:03, 3277.48 examples/s]

In [29]:
def run_metrics(pred, ref, num_samples=1000):    
    bleu = evaluate.load("sacrebleu")
    chrf = evaluate.load("chrf")

    preds = []
    refs = []

    for i in tqdm(range(num_samples), desc="🔹 Running inference"):
        preds.append(pred[i])
        refs.append([ref[i]])

        bleu.add(prediction=pred[i], reference=ref[i])
        chrf.add(prediction=pred[i], reference=ref[i])

    bleu_score = bleu.compute()
    chrf_score = chrf.compute()

    print("🔹 BLEU:", bleu_score)
    print("🔹 chrF++:", chrf_score)

    return preds, refs

In [30]:
google = pd.read_csv('/home/lilo/google_trans.csv', sep='\t', header=None)

predsg, refsg = run_metrics([i.replace('"', '') for i in google[0].tolist()], 
                            ref, num_samples=50)

🔹 Running inference: 100%|██████████| 50/50 [00:00<00:00, 17202.46it/s]

🔹 BLEU: {'score': 10.000153715755962, 'counts': [264, 119, 69, 44], 'totals': [1064, 1014, 964, 917], 'precisions': [24.81203007518797, 11.735700197238659, 7.157676348547718, 4.79825517993457], 'bp': 1.0, 'sys_len': 1064, 'ref_len': 644}
🔹 chrF++: {'score': 37.70556086026032, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [31]:
i=20

predsg[i], refsg[i]

("'6) Ветеринарный орган;',",
 [['6) уполномоченный орган в области ветеринарии;']])

In [24]:
google = pd.read_csv('/home/lilo/yandex_trans.csv', sep='\t', header=None)

predsg, refsg = run_metrics([i.replace('"', '') for i in google[0].tolist()], 
                            ref, num_samples=50)

🔹 Running inference: 100%|██████████| 50/50 [00:00<00:00, 13740.10it/s]

🔹 BLEU: {'score': 0.18867853638595608, 'counts': [25, 1, 0, 0], 'totals': [584, 534, 488, 444], 'precisions': [4.280821917808219, 0.18726591760299627, 0.10245901639344263, 0.05630630630630631], 'bp': 0.7235177953700749, 'sys_len': 584, 'ref_len': 773}
🔹 chrF++: {'score': 7.084208937314343, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [28]:
i=20

predsg[i], refsg[i]

('Шерил Гомес (Cheryl Gomez), сопредседатель',
 [['6) уполномоченный орган в области ветеринарии;']])

In [34]:
from train import load_data

import os
import pandas as pd
import polars as pl
import torch
import evaluate
from datasets import Dataset
import numpy as np
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer,DataCollatorForSeq2Seq


from tqdm import tqdm

model_dir='/home/lilo/experiments/t5small_kaz_rus/checkpoint-246'
test='/home/lilo/cleaned_data/test_dedup.txt'

test_set = load_data(test, 0.65)
tokenizer = T5Tokenizer.from_pretrained(model_dir)
model = T5ForConditionalGeneration.from_pretrained(model_dir)


In [37]:
def run_inference_t5(model, tokenizer, dataset, max_len=128, num_samples=100):
    model.eval()
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    bleu = evaluate.load("sacrebleu")
    chrf = evaluate.load("chrf")

    preds = []
    refs = []
    source = []

    for example in tqdm(dataset.select(range(num_samples)), desc="🔹 Running T5 Inference"):
        input_text = f"translate Kazakh to Russian: {example['kazakh']}"
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True, max_length=max_len).to(model.device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=max_len)

        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        ref = example["russian"]

        preds.append(pred)
        refs.append([ref])
        source.append(example["kazakh"])

        bleu.add(prediction=pred, reference=[ref])
        chrf.add(prediction=pred, reference=ref)

    bleu_score = bleu.compute()
    chrf_score = chrf.compute()

    print("🔹 BLEU:", bleu_score)
    print("🔹 chrF++:", chrf_score)

    return preds, refs, source


In [ ]:
from train import tokenize_data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)
dataset = tokenize_data(test_set, tokenizer, max_length=112, cache_dir=None)


🔹 Tokenizing dataset with cache file: None


Map: 100%|██████████| 63081/63081 [00:19<00:00, 3273.17 examples/s]


NameError: name 'run_inference_t5' is not defined

In [38]:
predsf, reff, sourcef = run_inference_t5(model, tokenizer, dataset, max_len=112, num_samples=1000) ###set len(test_set) for full metrics

🔹 Running T5 Inference: 100%|██████████| 1000/1000 [05:29<00:00,  3.04it/s]


🔹 BLEU: {'score': 6.109036231116707, 'counts': [3011, 1256, 762, 496], 'totals': [10062, 9101, 8153, 7274], 'precisions': [29.92446829656132, 13.80068124381936, 9.346252913038146, 6.818806708825956], 'bp': 0.47962794818394777, 'sys_len': 10062, 'ref_len': 17455}
🔹 chrF++: {'score': 13.375413701395331, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [39]:
from train import load_data

import os
import pandas as pd
import polars as pl
import torch
import evaluate
from datasets import Dataset
import numpy as np
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer,DataCollatorForSeq2Seq


from tqdm import tqdm

model_dir='/home/lilo/experiments/t5small_kaz_rus_filtered/checkpoint-177'
test='/home/lilo/cleaned_data/test_dedup.txt'

test_set = load_data(test, 0.65)
tokenizer = T5Tokenizer.from_pretrained(model_dir)
model = T5ForConditionalGeneration.from_pretrained(model_dir)


In [40]:
predsf, reff, sourcef = run_inference_t5(model, tokenizer, dataset, max_len=112, num_samples=1000) ###set len(test_set) for full metrics

🔹 Running T5 Inference: 100%|██████████| 1000/1000 [05:16<00:00,  3.16it/s]


🔹 BLEU: {'score': 6.779827364942663, 'counts': [3242, 1345, 828, 544], 'totals': [10459, 9475, 8501, 7599], 'precisions': [30.997227268381298, 14.195250659630608, 9.740030584637102, 7.158836689038031], 'bp': 0.5122729962946918, 'sys_len': 10459, 'ref_len': 17455}
🔹 chrF++: {'score': 14.296259097950776, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [46]:
light_code_switching_pairs = [
    {"kazakh": "Мен бүгін жұмысқа поездбен бардым.", "russian": "Сегодня я поехал на работу на поезде."},
    {"kazakh": "Біз магазинге барып, азық-түлік алдық.", "russian": "Мы пошли в магазин за продуктами."},
    {"kazakh": "Сенің другың жақсы адам екен.", "russian": "Твой друг оказался хорошим человеком."},
    {"kazakh": "Олар вечеринкада билеп жүрді.", "russian": "Они танцевали на вечеринке."},
    {"kazakh": "Мен работадан кеш шықтым.", "russian": "Я поздно вышел с работы."},
    {"kazakh": "Әкем машинасын жуып жатыр.", "russian": "Папа моет свою машину."},
    {"kazakh": "Бүгін кешке фильм көреміз бе?", "russian": "Посмотрим фильм сегодня вечером?"},
    {"kazakh": "Ол проблемаларды өзі шешеді.", "russian": "Он сам решает проблемы."},
    {"kazakh": "Ағам жаңа компьютер сатып алды.", "russian": "Мой брат купил новый компьютер."},
    {"kazakh": "Қазір пауза жасайық, сосын жалғастырамыз.", "russian": "Давайте сделаем паузу и потом продолжим."},
]

medium_code_switching_pairs = [
    {"kazakh": "Бүгін мен сені жәй ғана оставить хочу.", "russian": "Сегодня я просто хочу тебя оставить."},
    {"kazakh": "Ол всегда осылай істейді.", "russian": "Он всегда так делает."},
    {"kazakh": "Біз вчера кездестік, бірақ ол ештеңе айтпады.", "russian": "Мы встретились вчера, но он ничего не сказал."},
    {"kazakh": "Мына кітап мне очень понравился.", "russian": "Эта книга мне очень понравилась."},
    {"kazakh": "Олардың решение дұрыс болды.", "russian": "Их решение было правильным."},
    {"kazakh": "Мен қазір занят, кейін хабарласайын.", "russian": "Я сейчас занят, позже свяжусь."},
    {"kazakh": "Ол маған сказал, что всё хорошо.", "russian": "Он мне сказал, что всё хорошо."},
    {"kazakh": "Сен неге так долго келмедің?", "russian": "Почему ты так долго не приходил?"},
    {"kazakh": "Менің брат кеше машина сатып алды.", "russian": "Мой брат купил машину вчера."},
    {"kazakh": "Біз досымызға surprise жасадық.", "russian": "Мы устроили сюрприз другу."}
]

heavy_code_switching_pairs = [
    {"kazakh": "Мен бүгін ерте тұрдым, потому что надо было работать.", "russian": "Я встал рано сегодня, потому что нужно было работать."},
    {"kazakh": "Сен үй тапсырмасын орындадың ба? Или опять забыл?", "russian": "Ты сделал домашнее задание? Или снова забыл?"},
    {"kazakh": "Анам тамақ жасап жатыр, и я ей помогаю.", "russian": "Мама готовит еду, и я ей помогаю."},
    {"kazakh": "Бүгін кино көргім келеді, но не знаю что выбрать.", "russian": "Сегодня хочу посмотреть фильм, но не знаю, что выбрать."},
    {"kazakh": "Кешке кездесеміз, если не будет дождя.", "russian": "Встретимся вечером, если не будет дождя."},
    {"kazakh": "Сенің досың келді ме? Потому что мы его ждём.", "russian": "Твой друг пришёл? Потому что мы его ждём."},
    {"kazakh": "Мен шаршадым, давай завтра продолжим.", "russian": "Я устал, давай продолжим завтра."},
    {"kazakh": "Жақсы, бастайық. А потом объясню всё.", "russian": "Хорошо, начнём. А потом всё объясню."},
    {"kazakh": "Ол кетті, потому что устал.", "russian": "Он ушёл, потому что устал."},
    {"kazakh": "Менің телефоным өшіп қалды, и я не смог позвонить.", "russian": "Мой телефон выключился, и я не смог позвонить."}
]




In [48]:
def translate_kazakh_text(model, tokenizer, text, max_len=128):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    input_text = f"translate Kazakh to Russian: {text}"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding=True, max_length=max_len).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_len)

    translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated


def comp_chrf(model, tokenizer, light_code_switching_pairs):
    chrf = evaluate.load("chrf")
    for i in tqdm(light_code_switching_pairs):
        translated = translate_kazakh_text(model, tokenizer, i['kazakh'])
        chrf.add(prediction=translated, reference=i['russian'])
    chrf_score = chrf.compute()
    print("🔹 chrF++:", chrf_score)

comp_chrf(model, tokenizer, light_code_switching_pairs)
comp_chrf(model, tokenizer, medium_code_switching_pairs)
comp_chrf(model, tokenizer, heavy_code_switching_pairs)

100%|██████████| 10/10 [00:01<00:00,  6.56it/s]


🔹 chrF++: {'score': 8.018479801169375, 'char_order': 6, 'word_order': 0, 'beta': 2}


100%|██████████| 10/10 [00:01<00:00,  6.36it/s]


🔹 chrF++: {'score': 12.265797362964879, 'char_order': 6, 'word_order': 0, 'beta': 2}


100%|██████████| 10/10 [00:01<00:00,  5.60it/s]

🔹 chrF++: {'score': 16.319372108177447, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [49]:
from train import load_data

import os
import pandas as pd
import polars as pl
import torch
import evaluate
from datasets import Dataset
import numpy as np
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer,DataCollatorForSeq2Seq


from tqdm import tqdm

model_dir='/home/lilo/experiments/t5small_kaz_rus/checkpoint-246'
test='/home/lilo/cleaned_data/test_dedup.txt'

test_set = load_data(test, 0.65)
tokenizer = T5Tokenizer.from_pretrained(model_dir)
model = T5ForConditionalGeneration.from_pretrained(model_dir)

comp_chrf(model, tokenizer, light_code_switching_pairs)
comp_chrf(model, tokenizer, medium_code_switching_pairs)
comp_chrf(model, tokenizer, heavy_code_switching_pairs)

100%|██████████| 10/10 [00:01<00:00,  6.63it/s]


🔹 chrF++: {'score': 7.206096594338546, 'char_order': 6, 'word_order': 0, 'beta': 2}


100%|██████████| 10/10 [00:01<00:00,  6.54it/s]


🔹 chrF++: {'score': 10.185070642837658, 'char_order': 6, 'word_order': 0, 'beta': 2}


100%|██████████| 10/10 [00:02<00:00,  3.76it/s]

🔹 chrF++: {'score': 15.453251799683748, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [55]:
from train import load_data

import os
import pandas as pd
import polars as pl
import torch
import evaluate
from datasets import Dataset
import numpy as np
from transformers import (
    M2M100Tokenizer,
    M2M100ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from tqdm import tqdm

model_dir='/home/lilo/experiments/exp_m2m100_065_kz_rus/checkpoint-1000'
test='/home/lilo/cleaned_data/test_dedup.txt'

test_set = load_data(test, 0.65)
tokenizer = M2M100Tokenizer.from_pretrained(model_dir)
model = M2M100ForConditionalGeneration.from_pretrained(model_dir)
comp_chrf(model, tokenizer, light_code_switching_pairs)
comp_chrf(model, tokenizer, medium_code_switching_pairs)
comp_chrf(model, tokenizer, heavy_code_switching_pairs)

100%|██████████| 10/10 [00:05<00:00,  1.84it/s]


🔹 chrF++: {'score': 47.22848912978698, 'char_order': 6, 'word_order': 0, 'beta': 2}


100%|██████████| 10/10 [00:01<00:00,  6.48it/s]


🔹 chrF++: {'score': 58.720972170623455, 'char_order': 6, 'word_order': 0, 'beta': 2}


100%|██████████| 10/10 [00:01<00:00,  5.26it/s]

🔹 chrF++: {'score': 60.383025717610316, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [54]:
from train import load_data

import os
import pandas as pd
import polars as pl
import torch
import evaluate
from datasets import Dataset
import numpy as np
from transformers import (
    M2M100Tokenizer,
    M2M100ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from tqdm import tqdm

model_dir='/home/lilo/experiments/exp_m2m100_filtered065_kz_rus/checkpoint-1131'
test='/home/lilo/cleaned_data/test_dedup.txt'

test_set = load_data(test, 0.65)
tokenizer = M2M100Tokenizer.from_pretrained(model_dir)
model = M2M100ForConditionalGeneration.from_pretrained(model_dir)
comp_chrf(model, tokenizer, light_code_switching_pairs)
comp_chrf(model, tokenizer, medium_code_switching_pairs)
comp_chrf(model, tokenizer, heavy_code_switching_pairs)

100%|██████████| 10/10 [00:01<00:00,  5.33it/s]


🔹 chrF++: {'score': 48.77453820985645, 'char_order': 6, 'word_order': 0, 'beta': 2}


100%|██████████| 10/10 [00:01<00:00,  6.49it/s]


🔹 chrF++: {'score': 60.31904677702645, 'char_order': 6, 'word_order': 0, 'beta': 2}


100%|██████████| 10/10 [00:01<00:00,  5.41it/s]

🔹 chrF++: {'score': 60.57035834951747, 'char_order': 6, 'word_order': 0, 'beta': 2}
